In [1]:
import pandas as pd
import mlflow
import mlflow.catboost
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://localhost:5000")
client = MlflowClient()

REGISTERED_MODEL_NAME = "atm_placement_catboost"

# --- Вариант найти версию по тегу PRD
prd_versions = [
    v for v in client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
    if v.tags.get("PRD") == "true"]

assert prd_versions, "Не найдено ни одной версии с тегом PRD=true"
prd_version = prd_versions[0]
print(f"Найдена PRD-версия по тегу: v{prd_version.version}")
print("Теги:", prd_version.tags)

# Загрузка модели (по алиасу PRD)
model_uri = f"models:/{REGISTERED_MODEL_NAME}@PRD"
model = mlflow.catboost.load_model(model_uri)
print("\nМодель загружена из:", model_uri)

# список фич, которые модель ожидает на вход (в правильном порядке)
feature_cols = list(model.feature_names_)
print("Число фич у модели:", len(feature_cols))

Найдена PRD-версия по тегу: v1
Теги: {'PRD': 'true', 'cv_roc_auc': '0.8733', 'cv_pr_auc': '0.8698', 'strategy': 'smart_negatives'}

Модель загружена из: models:/atm_placement_catboost@PRD
Число фич у модели: 90


In [2]:
# берём небольшой кусок исходных данных как "новые" объекты
df_new = pd.read_csv("grid_features.csv", nrows=2000)

# готовим признаки X строго в том же наборе и порядке, что у модели
X_new = df_new[feature_cols].fillna(0)

# предсказание: вероятность класса 1 (есть банкомат)
df_new["atm_proba"] = model.predict_proba(X_new)[:, 1]

# топ-10 ячеек по вероятности разместить банкомат
top10 = df_new.nlargest(10, "atm_proba")[["city", "lat", "lon", "atm_proba"]]
print("Тестовый предикт PRD-модели, т.е. топ-10 ячеек по вероятности:")
print(top10.to_string(index=False))

Тестовый предикт PRD-модели, т.е. топ-10 ячеек по вероятности:
  city       lat       lon  atm_proba
Москва 54.974912 37.416247   0.566466
Москва 54.988386 37.328746   0.455233
Москва 54.983895 37.328746   0.424704
Москва 54.974912 37.376474   0.392153
Москва 54.988386 37.320791   0.387816
Москва 54.974912 37.408292   0.386427
Москва 54.974912 37.368519   0.380026
Москва 54.974912 37.996937   0.359848
Москва 54.988386 37.312837   0.355357
Москва 54.970420 37.384429   0.351837
